In [1]:
import polars as pl

In [4]:
import pandas as pd

# Read the complete SKS dataset
df = pd.read_fwf(
    "SKScomplete.txt",
    widths=[3, 20, 8, 8, 8, 120, 47],
    names=[
        "category",
        "code",
        "valid_from",
        "modified",
        "valid_until",
        "name",
        "other"
    ],
    dtype=str,
    encoding="latin-1"
)


In [5]:
df

,category,code,valid_from,modified,valid_until,name,other
0,sad,mA,2002010,12002043,02500010,1Administrative forhold,SKS 1
1,adm,AA,20020101,20021031,25000101,Patientadministrative forhold,SKS 1
2,adm,AAF,20020101,20021031,25000101,Booking,SKS 1
3,adm,AAF1,20030401,20030331,25000101,Indlæggelse,SKS 1
4,adm,AAF11,20040101,20040331,25000101,Genindlæggelse,SKS 1
...,...,...,...,...,...,...,...
107490,und,UXZ73,20000101,20001027,20001031,med vægtskanning,SKS 3
107491,und,UXZ74,20000101,20001027,20001031,med bestemmelse af fosterstilling,SKS 3
107492,und,UXZ75,20000101,20001027,20001031,med nakkefoldsskanning,SKS 3
107493,und,UXZ90,20000101,20001027,20001031,med funktionsundersøgelse,SKS 3


In [6]:
# Keep all codes, regardless of classification
df = df[["code", "name"]]

# Remove empty entries
df = df.dropna(subset=["code", "name"])

# Remove duplicates
df = df.drop_duplicates()

# Export
df.to_csv(
    "all_sks_mappings.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Exported {len(df):,} mappings")

# Example lookup
print(df[df["code"].str.startswith("DC50")])


Exported 102,102 mappings
         code                                               name
18263    DC50                                      Kræft i bryst
18264    DC50                                         Brystkræft
18265   DC500               Neopl mal papillae et areolae mammae
18266   DC500             Kræft i brystvorte eller areola mammae
18267  DC500A                           Neopl mal mammae, areola
18268  DC500A                             Kræft i areolae mammae
18269  DC500B                          Neopl mal mammae, papilla
18270  DC500B                                 Kræft i brystvorte
18271  DC500C                             Paget's sygdom i mamma
18272  DC500D                      Paget's disease of the nipple
18273  DC500M  Kræft i brystvorte eller areola mammae med met...
18274   DC501                      Neopl mal mammae centrale del
18275   DC501             Brystkræft i den centrale del af mamma
18276  DC501M  Brystkræft i centrale del af mamma med metastaser

In [7]:
table_path = "/Users/mariaelkjaermontgomery/Library/CloudStorage/Dropbox/phd/code/ehr2meds/data/MEDS_to_text/nkn-tabellen.xlsx"

nkn = pl.read_excel(table_path, sheet_name="Ark1")

# NPU (and other) Code -> ShortName
npu_to_shortname = (
    nkn.select("Code", "ShortName")
    .drop_nulls(["Code", "ShortName"])
    .unique(subset=["Code"], keep="first")
).rename({"Code": "code", "ShortName": "name"})

npu_mapping = dict(npu_to_shortname.iter_rows())

npu_to_shortname.write_csv("all_npu_mappings.csv")
print(f"Exported {npu_to_shortname.height:,} mappings to all_npu_mappings.csv")
print(npu_mapping.get("NPU17074"))  # e.g. 'Acanthocytter;B'

Exported 12,015 mappings to all_npu_mappings.csv
Acanthocytter;B


In [11]:
sks_df = pl.read_csv("all_sks_mappings.csv")

atc_codes = (
    sks_df
    .filter(pl.col("code").str.starts_with("M"))
    .with_columns(code=pl.col("code").str.slice(1))  # drop only the leading M
    .filter(pl.col("code").str.strip_chars() != "")
)

atc_codes.write_csv("all_atc_mappings.csv")
print(f"Exported {atc_codes.height:,} mappings to all_atc_mappings.csv")
atc_codes.head()

Exported 12,136 mappings to all_atc_mappings.csv


code,name
str,str
"""A""","""Fordøjelsesorganer og stofskif…"
"""A01""","""Stomatological preparations"""
"""A01""","""midler mod sygdomme i mundhule…"
"""A01A""","""Stomatological preparations"""
"""A01A""","""stomatologica"""
